# S11 · Pipeline de limpieza y transformación

NovaMarket · Preprocesamiento de Datos · Grupo 05

Integrantes: María Andrea Torres · Ricardo Borja · Gabriel Murillo Velandia · Laura Marcela Gómez

---

Transformaciones variables de fecha y de texto, discretización de la edad y corrección del sesgo del monto

In [5]:
!pip install numpy pandas scikit-learn

In [6]:
# --- Configuración única del pipeline -------------------------------------------------
import gc
import os
import re
import unicodedata

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder,
                                   PowerTransformer, RobustScaler, StandardScaler,
                                   TargetEncoder)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

ENTRADA  = "S04_PD_Borja_DatasetImputacionInicial.csv"   # único archivo de entrada
SALIDA   = "S11_PD_Grupo05_DatasetTransformado.csv"        # único archivo de salida
OBJETIVO = "fuga_cliente"
SEMILLA  = 42

CATEGORICAS = ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]
NUMERICAS   = ["precio_unitario", "unidades_vendidas", "monto_compra",
               "edad_cliente", "peso_pedido_kg"]

# La Semana 6 leía todo como texto para poder comparar variantes de escritura sin que
# pandas adivinara tipos por el camino. Se conserva igual.
df = pd.read_csv(ENTRADA, dtype=str)
filas_inicio = len(df)

pd.DataFrame({"valor": [ENTRADA, filas_inicio, df.shape[1]]},
             index=["archivo de entrada", "filas", "columnas"])

,valor
archivo de entrada,S04_PD_Borja_DatasetImputacionInicial.csv
filas,620000
columnas,27


---
## Etapa 1: Limpieza de inconsistencias y duplicados

*Viene de `S06_PD_Borja_LimpiezaInconsistencias.ipynb`.* Tres problemas distintos: la misma ciudad
escrita de varias formas, la fecha guardada en dos formatos incompatibles, y filas repetidas con
distinto `id_pedido`. Se resuelven en ese orden porque el tercero depende de los dos primeros: no se
pueden comparar dos filas como duplicadas si sus categorías están escritas distinto.

In [7]:
# Unificación de la escritura: primero la regla general (espacios y mayúsculas), después el
# diccionario de las variantes que la regla general no alcanza a juntar (tildes y PayPal).
diagnostico = pd.DataFrame({"variantes_antes": [df[c].nunique(dropna=True) for c in CATEGORICAS]},
                           index=CATEGORICAS)

MAPA_CORRECCION = {
    "categoria_producto": {"Electronica": "Electrónica", "Jugueteria": "Juguetería"},
    "metodo_pago":        {"Paypal": "PayPal"},
}
for c in CATEGORICAS:
    df[c] = df[c].str.strip().str.title().replace(MAPA_CORRECCION.get(c, {}))

TEXTO_LIBRE = ["producto", "comentario_cliente", "nivel_satisfaccion", "nivel_lealtad"]
for c in TEXTO_LIBRE:
    df[c] = df[c].str.strip()

diagnostico["categorias_despues"] = [df[c].nunique(dropna=True) for c in CATEGORICAS]
diagnostico["nulos"] = [int(df[c].isna().sum()) for c in CATEGORICAS]
diagnostico

,variantes_antes,categorias_despues,nulos
ciudad_tienda,12,12,0
categoria_producto,32,6,0
canal_compra,20,4,0
metodo_pago,20,4,0


In [8]:
# La fecha viene en dos formatos: ISO y con barras en orden día/mes/año (se comprobó en la
# Semana 6 que el primer campo llega a 31 y el segundo nunca pasa de 12). Las fechas de
# calendario imposible se marcan en una bandera y la fila se conserva.
texto_fecha = df["fecha_pedido"].str.strip()
es_iso    = texto_fecha.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
es_barras = texto_fecha.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)
es_vacia  = texto_fecha.isna()

fecha = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
fecha[es_iso]    = pd.to_datetime(texto_fecha[es_iso], format="%Y-%m-%d", errors="coerce")
fecha[es_barras] = pd.to_datetime(texto_fecha[es_barras], dayfirst=True, errors="coerce")

df["fecha_pedido_invalida"] = fecha.isna() & ~es_vacia
df["fecha_pedido"] = fecha

pd.DataFrame({"filas": [int(es_iso.sum()), int(es_barras.sum()), int(es_vacia.sum()),
                        int(df["fecha_pedido_invalida"].sum()),
                        int(df["fecha_pedido"].isna().sum())]},
             index=["formato ISO", "formato con barras", "vacías de origen",
                    "calendario imposible", "sin fecha al final"])

,filas
formato ISO,199608
formato con barras,414858
vacías de origen,5534
calendario imposible,16466
sin fecha al final,22000


In [9]:
# Duplicados. El id_pedido se excluye de la comparación: dos filas idénticas en las otras 27
# columnas son el mismo pedido cargado dos veces aunque el identificador cambie. Se conserva
# la versión con la actualización de stock más reciente.
sin_id = [c for c in df.columns if c != "id_pedido"]
antes_regla = len(df)

df = (df.sort_values("fecha_actualizacion_stock", ascending=False, kind="mergesort")
        .drop_duplicates(subset=sin_id, keep="first")
        .sort_index())

pd.DataFrame({"filas": [antes_regla, len(df), antes_regla - len(df), int(df.duplicated().sum())]},
             index=["antes de la regla", "después de la regla", "eliminadas",
                    "duplicados exactos restantes"])

,filas
antes de la regla,620000
después de la regla,620000
eliminadas,0
duplicados exactos restantes,0


In [10]:
# AJUSTE DE ENCADENAMIENTO (1 de 2)
# En los cuadernos semanales este paso lo hacía el CSV: la Semana 6 escribía el archivo y la
# Semana 7 lo releía, y en esa relectura pandas volvía a inferir los tipos. Sin archivo
# intermedio hay que hacer esa conversión de forma explícita, o las etapas siguientes
# recibirían números guardados como texto. No cambia ningún valor, solo el tipo.
ENTERAS = ["codigo_postal", "unidades_vendidas", "edad_cliente", OBJETIVO]
FLOTANTES = ["precio_unitario", "monto_compra", "peso_pedido_kg"]
BANDERAS = ["ciudad_tienda_imputada", "satisfaccion_sin_dato", "correo_sin_dato",
            "sin_comentario", "precio_sin_dato", "monto_sin_dato", "fecha_pedido_sin_dato"]

for c in ENTERAS + FLOTANTES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in BANDERAS:
    df[c] = df[c].map({"True": True, "False": False}).astype(bool)

pd.DataFrame({"tipo": [str(df[c].dtype) for c in ENTERAS + FLOTANTES + BANDERAS[:2]]},
             index=ENTERAS + FLOTANTES + BANDERAS[:2])

,tipo
codigo_postal,int64
unidades_vendidas,int64
edad_cliente,int64
fuga_cliente,int64
precio_unitario,float64
monto_compra,float64
peso_pedido_kg,float64
ciudad_tienda_imputada,bool
satisfaccion_sin_dato,bool


---
## Etapa 2: Tratamiento de valores atípicos

*Viene de `S07_PD_Borja_TratamientoOutliers.ipynb`.* El hallazgo que ordena esta etapa es que
`monto_compra = precio_unitario × unidades_vendidas` se cumple en el 100 % de las filas evaluables.
Por eso no se toca el monto directamente: se limpian los dos factores y el monto se **recalcula**.
Ninguna fila se elimina; se anulan valores puntuales y cada anulación queda registrada en una
bandera para poder rastrearla después.

In [11]:
# Tres capas de problema en los factores: valores negativos (imposibles), centinelas de dos
# niveles (9999 y 5000 en unidades, 200 y 150 en edad, 999 millones y 50 millones en precio) y
# el cero de edad. Cada valor anulado deja su bandera.
REGLAS = {
    "precio_unitario":   lambda s: (s < 0) | s.isin([999000000.0, 50000000.0]),
    "unidades_vendidas": lambda s: (s < 0) | s.isin([9999, 5000]),
    "edad_cliente":      lambda s: (s < 0) | s.isin([200, 150]) | (s == 0),
}

resumen = []
for col, regla in REGLAS.items():
    marca = regla(df[col]).fillna(False)
    df[col + "_atipico"] = marca
    df.loc[marca, col] = np.nan
    resumen.append({"variable": col, "valores anulados": int(marca.sum()),
                    "% del total": round(100 * marca.mean(), 2)})

pd.DataFrame(resumen).set_index("variable")

,valores anulados,% del total
variable,,
precio_unitario,25000,4.03
unidades_vendidas,35931,5.80
edad_cliente,36000,5.81


In [12]:
# El monto se recalcula desde los factores ya limpios, y el peso se capa en su percentil 99
# (el bloque alto son pedidos de 900 kg o más, sin un solo caso entre 15 y 900 kg: es un
# artefacto de carga, no una cola real).
monto_previo = df["monto_compra"].copy()
df["monto_compra"] = df["precio_unitario"] * df["unidades_vendidas"]
df["monto_atipico"] = df["monto_compra"].isna() & monto_previo.notna()

p99_peso = df["peso_pedido_kg"].quantile(0.99)
df["peso_pedido_kg_original"] = df["peso_pedido_kg"]
df["peso_pedido_kg"] = df["peso_pedido_kg"].clip(upper=p99_peso)
df["peso_capado"] = df["peso_pedido_kg_original"] > p99_peso

control_etapa2 = pd.DataFrame({"valor": [
    round(float(monto_previo.max()), 2), round(float(df["monto_compra"].max()), 2),
    int((df["monto_compra"] < 0).sum()), int(df["monto_atipico"].sum()),
    round(float(p99_peso), 2), int(df["peso_capado"].sum()), len(df),
]}, index=["monto máximo antes", "monto máximo después", "montos negativos restantes",
           "montos anulados por arrastre", "percentil 99 del peso", "pesos capados",
           "filas (no se elimina ninguna)"])
del monto_previo
gc.collect()
control_etapa2

,valor
monto máximo antes,9.989001e+12
monto máximo después,4.457500e+06
montos negativos restantes,0.000000e+00
montos anulados por arrastre,5.829100e+04
percentil 99 del peso,1.298200e+03
pesos capados,6.197000e+03
filas (no se elimina ninguna),6.200000e+05


---
## Etapa 3: Escalado de variables numéricas

*Viene de `S08_PD_Grupo05_Escalado.ipynb`.* Cada escalador se ajusta **solo con el conjunto de
entrenamiento** y después transforma los dos conjuntos. Ese orden es lo que evita la fuga de datos:
si el escalador viera la prueba, sus parámetros llevarían información que en producción no existiría.

La técnica se eligió por columna, no por comodidad: Min-Max donde el rango es acotado y estable,
Z-score donde la distribución es simétrica, y escalado robusto donde hay atípicos legítimos o la
distribución es bimodal.

In [13]:
# AJUSTE DE ENCADENAMIENTO (2 de 2)
# La Semana 8 partía el dataset en dos objetos (train y test), los escalaba por separado y los
# volvía a unir ordenados por índice antes de exportar. Aquí se parte el índice en vez del
# DataFrame y se marca la partición en la columna 'conjunto': el resultado fila a fila es
# idéntico (misma semilla, misma proporción) y se evita duplicar el dataset en memoria.
idx_train, idx_test = train_test_split(df.index, test_size=0.20, random_state=SEMILLA)
es_train = df.index.isin(idx_train)
df["conjunto"] = np.where(es_train, "entrenamiento", "prueba")

PLAN = {
    "precio_unitario":   MinMaxScaler(),      # rango acotado y estable
    "unidades_vendidas": MinMaxScaler(),      # discreta, seis niveles
    "monto_compra":      RobustScaler(),      # atípicos legítimos, asimetría 2,0
    "edad_cliente":      StandardScaler(),    # simétrica
    "peso_pedido_kg":    RobustScaler(),      # atípicos y bimodal
}

escaladores = {}
for col, escalador in PLAN.items():
    escalador.fit(df.loc[es_train, [col]])                 # ajuste: solo entrenamiento
    df[col + "_esc"] = escalador.transform(df[[col]])      # transformación: los dos conjuntos
    escaladores[col] = escalador

pd.DataFrame({
    "escalador": [type(e).__name__ for e in escaladores.values()],
    "ajustado con": ["solo entrenamiento"] * len(escaladores),
    "nulos que se conservan": [int(df[c + "_esc"].isna().sum()) for c in PLAN],
}, index=list(PLAN))

,escalador,ajustado con,nulos que se conservan
precio_unitario,MinMaxScaler,solo entrenamiento,46000
unidades_vendidas,MinMaxScaler,solo entrenamiento,35931
monto_compra,RobustScaler,solo entrenamiento,79291
edad_cliente,StandardScaler,solo entrenamiento,36000
peso_pedido_kg,RobustScaler,solo entrenamiento,0


In [14]:
# Verificación de que no hubo fuga: el parámetro guardado por cada escalador tiene que ser el
# estadístico del entrenamiento, no el del dataset completo.
chequeo = []
for col, e in escaladores.items():
    if isinstance(e, MinMaxScaler):
        guardado = (e.data_min_[0], e.data_max_[0])
        de_train = (df.loc[es_train, col].min(), df.loc[es_train, col].max())
        de_todo  = (df[col].min(), df[col].max())
    elif isinstance(e, StandardScaler):
        guardado = (e.mean_[0], e.scale_[0])
        de_train = (df.loc[es_train, col].mean(), df.loc[es_train, col].std(ddof=0))
        de_todo  = (df[col].mean(), df[col].std(ddof=0))
    else:
        q1t, q3t = df.loc[es_train, col].quantile([0.25, 0.75])
        q1a, q3a = df[col].quantile([0.25, 0.75])
        guardado = (e.center_[0], e.scale_[0])
        de_train = (df.loc[es_train, col].median(), q3t - q1t)
        de_todo  = (df[col].median(), q3a - q1a)
    chequeo.append({"variable": col, "escalador": type(e).__name__,
                    "coincide con entrenamiento": bool(np.allclose(guardado, de_train)),
                    "coincide con dataset completo": bool(np.allclose(guardado, de_todo))})

tabla_chequeo = pd.DataFrame(chequeo).set_index("variable")
display(tabla_chequeo)

pd.DataFrame({"valor": [
    int(tabla_chequeo["coincide con entrenamiento"].sum()), len(tabla_chequeo),
    "sin fuga" if tabla_chequeo["coincide con entrenamiento"].all() else "revisar",
    int(es_train.sum()), int((~es_train).sum()),
]}, index=["parámetros que coinciden con el entrenamiento", "variables evaluadas", "lectura",
           "filas de entrenamiento", "filas de prueba"])

,escalador,coincide con entrenamiento,coincide con dataset completo
variable,,,
precio_unitario,MinMaxScaler,True,True
unidades_vendidas,MinMaxScaler,True,True
monto_compra,RobustScaler,True,False
edad_cliente,StandardScaler,True,False
peso_pedido_kg,RobustScaler,True,True


,valor
parámetros que coinciden con el entrenamiento,5
variables evaluadas,5
lectura,sin fuga
filas de entrenamiento,496000
filas de prueba,124000


---
## Etapa 4: Codificación de variables categóricas

*Viene de `S09_PD_Borja_Codificacion.ipynb`.* Cuatro técnicas, cada una para un problema distinto:
one-hot para las nominales de cardinalidad baja, ordinal con orden declarado para las dos que sí
tienen jerarquía real, y frecuencia más target encoding para `codigo_postal`, que con 493 categorías
reventaría cualquier one-hot. Todos los codificadores se ajustan solo con entrenamiento, igual que
los escaladores.

In [15]:
# Dos arreglos previos heredados del diagnóstico: normalizar nivel_satisfaccion (diez variantes
# de escritura para tres niveles) y sacar el dominio del correo, que es la única categoría real
# escondida dentro de un identificador.
df["nivel_satisfaccion"] = df["nivel_satisfaccion"].str.strip().str.capitalize()
df.loc[df["nivel_satisfaccion"] == "Sin dato", "nivel_satisfaccion"] = np.nan

correo = df["correo_cliente"].str.strip().str.lower()
tiene_arroba  = correo.str.contains("@", na=False)
formato_punto = ~tiene_arroba & correo.str.match(r"^cliente\d+\.", na=False)

df["dominio_correo"] = "desconocido"
df.loc[tiene_arroba, "dominio_correo"] = correo[tiene_arroba].str.split("@").str[-1]
df.loc[formato_punto, "dominio_correo"] = correo[formato_punto].str.replace(
    r"^cliente\d+\.", "", regex=True)
del correo, tiene_arroba, formato_punto

pd.DataFrame({"valor": [
    sorted(df["nivel_satisfaccion"].dropna().unique().tolist()),
    int(df["nivel_satisfaccion"].isna().sum()),
    bool((df["nivel_satisfaccion"].isna() == df["satisfaccion_sin_dato"]).all()),
    int(df["dominio_correo"].nunique()),
    bool(((df["dominio_correo"] == "desconocido") == df["correo_sin_dato"]).all()),
]}, index=["niveles de satisfacción que quedan", "nulos tras normalizar",
           "los nulos coinciden con su bandera", "dominios de correo distintos",
           "'desconocido' coincide con su bandera"])

,valor
niveles de satisfacción que quedan,"[Alto, Bajo, Medio]"
nulos tras normalizar,45000
los nulos coinciden con su bandera,True
dominios de correo distintos,6
'desconocido' coincide con su bandera,True


In [16]:
# One-hot para las siete nominales de cardinalidad baja, y ordinal con el orden declarado a mano
# para las dos jerárquicas. El orden importa: alfabéticamente Oro quedaría por debajo de Plata y
# la relación con la fuga se rompería.
es_train = df["conjunto"] == "entrenamiento"
es_test  = df["conjunto"] == "prueba"

NOMINALES_OH = ["canal_compra", "metodo_pago", "categoria_producto", "ciudad_tienda",
                "comentario_cliente", "dominio_correo", "producto"]
ORDENES = {"nivel_lealtad":      ["Bronce", "Plata", "Oro", "Platino"],
           "nivel_satisfaccion": ["Bajo", "Medio", "Alto"]}
ORDINALES = list(ORDENES)

def limpiar_nombre(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()

onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.uint8)
onehot.fit(df.loc[es_train, NOMINALES_OH])                      # ajuste: solo entrenamiento
X_oh = pd.DataFrame(onehot.transform(df[NOMINALES_OH]),         # transformación: los dos conjuntos
                    columns=[limpiar_nombre(n) for n in onehot.get_feature_names_out(NOMINALES_OH)],
                    index=df.index)

ordinal = OrdinalEncoder(categories=[ORDENES[c] for c in ORDINALES],
                         handle_unknown="use_encoded_value", unknown_value=-1, dtype=float)
ordinal.fit(df.loc[es_train, ORDINALES])
X_ord = pd.DataFrame(ordinal.transform(df[ORDINALES]),
                     columns=[c + "_ord" for c in ORDINALES], index=df.index)
for c in ORDINALES:      # un nulo no es una categoría desconocida: se conserva como nulo
    X_ord.loc[df[c].isna(), c + "_ord"] = np.nan

pd.DataFrame({"columnas generadas": [len(c) for c in onehot.categories_]},
             index=NOMINALES_OH).assign(
    categorias_en_prueba=[df.loc[es_test, c].nunique() for c in NOMINALES_OH])

,columnas generadas,categorias_en_prueba
canal_compra,4,4
metodo_pago,4,4
categoria_producto,6,6
ciudad_tienda,12,12
comentario_cliente,13,13
dominio_correo,6,6
producto,24,24


In [17]:
# codigo_postal: 493 categorías, de las cuales la mayoría tiene menos del 0,05 % de las filas.
# Los raros se agrupan en 'Otro' (lista decidida mirando solo entrenamiento), la frecuencia se
# aprende en entrenamiento, y el target encoding usa cross fitting para que ninguna fila se
# codifique con su propio objetivo.
UMBRAL_RARA = 0.0005
frec_train = df.loc[es_train, "codigo_postal"].value_counts()
codigos_raros = frec_train[frec_train < UMBRAL_RARA * es_train.sum()].index

df["codigo_postal_agrupado"] = df["codigo_postal"].astype(str)
df.loc[df["codigo_postal"].isin(codigos_raros), "codigo_postal_agrupado"] = "Otro"

frecuencia = df.loc[es_train, "codigo_postal"].value_counts(normalize=True)
df["codigo_postal_freq"] = df["codigo_postal"].map(frecuencia).fillna(0.0)

te = TargetEncoder(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=SEMILLA)
X_te = pd.Series(np.nan, index=df.index, name="codigo_postal_te")
X_te.loc[es_train] = te.fit_transform(df.loc[es_train, ["codigo_postal_agrupado"]],
                                      df.loc[es_train, OBJETIVO]).ravel()
X_te.loc[es_test]  = te.transform(df.loc[es_test, ["codigo_postal_agrupado"]]).ravel()
df["codigo_postal_te"] = X_te
del X_te

pd.DataFrame({"valor": [
    int(df.loc[es_train, "codigo_postal"].nunique()), len(codigos_raros),
    int(df.loc[es_train, "codigo_postal_agrupado"].nunique()),
    round(float(df.loc[es_train, "codigo_postal_te"].mean()), 4),
    round(float(df.loc[es_train, OBJETIVO].mean()), 4),
]}, index=["códigos distintos en entrenamiento", "códigos agrupados en 'Otro'",
           "categorías tras agrupar", "media del target encoding en entrenamiento",
           "tasa de fuga en entrenamiento (deben parecerse)"])

,valor
códigos distintos en entrenamiento,493.0000
códigos agrupados en 'Otro',263.0000
categorías tras agrupar,231.0000
media del target encoding en entrenamiento,0.2093
tasa de fuga en entrenamiento (deben parecerse),0.2093


In [18]:
# Consolidación de la etapa: se retiran las columnas de texto que ya quedaron codificadas y se
# pegan las nuevas. El comentario se guarda aparte porque la Etapa 5 lo necesita como texto:
# en los cuadernos semanales se recuperaba releyendo el CSV de la Semana 8.
comentarios = df[["id_pedido", "comentario_cliente"]].copy()

TEXTO_CODIFICADO = NOMINALES_OH + ORDINALES + ["codigo_postal", "codigo_postal_agrupado"]
df = pd.concat([df.drop(columns=TEXTO_CODIFICADO), X_oh, X_ord], axis=1)
df = df[[c for c in df.columns if c not in (OBJETIVO, "conjunto")] + [OBJETIVO, "conjunto"]]
del X_oh, X_ord
gc.collect()

texto_restante = [c for c in df.columns if df[c].dtype == object]
pd.DataFrame({"valor": [
    len(TEXTO_CODIFICADO), df.shape[1], ", ".join(texto_restante),
]}, index=["columnas de texto retiradas", "columnas tras la etapa 4",
           "columnas de texto que quedan (identificadores y la partición)"])

,valor
columnas de texto retiradas,11
columnas tras la etapa 4,104
columnas de texto que quedan (identificadores y la partición),"id_pedido, id_cliente, correo_cliente, fecha_a..."


---
## Etapa 5: Transformaciones de fecha, texto y variables sesgadas

*Viene de `S10_PD_Borja_Transformaciones.ipynb`.* Tres cosas distintas que comparten un mismo
propósito: sacar información que ya estaba en el dataset pero que ningún modelo podía usar tal como
venía. De la fecha salen cuatro variables de calendario, del comentario cuatro variables de texto,
la edad se discretiza en grupos con sentido de negocio, y el monto recibe dos correcciones de sesgo
(logaritmo y Box-Cox, esta última con el lambda aprendido solo en entrenamiento).

In [19]:
# Se recupera el comentario y se derivan las cuatro variables de calendario. Los nulos de fecha
# se propagan como nulos: no se inventa un mes para un pedido sin fecha.
df = df.merge(comentarios, on="id_pedido", how="left", validate="one_to_one")
del comentarios
gc.collect()

df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")
df["mes_pedido"]           = df["fecha_pedido"].dt.month.astype("Int64")
df["trimestre_pedido"]     = df["fecha_pedido"].dt.quarter.astype("Int64")
df["dia_semana_pedido"]    = df["fecha_pedido"].dt.dayofweek.astype("Int64")
df["es_fin_semana_pedido"] = (df["fecha_pedido"].dt.dayofweek >= 5).astype("Int64")
df.loc[df["fecha_pedido"].isna(), "es_fin_semana_pedido"] = pd.NA

TEMPORALES = ["mes_pedido", "trimestre_pedido", "dia_semana_pedido", "es_fin_semana_pedido"]
pd.DataFrame({
    "tipo": [str(df[c].dtype) for c in TEMPORALES],
    "mínimo": [df[c].min() for c in TEMPORALES],
    "máximo": [df[c].max() for c in TEMPORALES],
    "nulos": [int(df[c].isna().sum()) for c in TEMPORALES],
    "los nulos coinciden con las fechas nulas":
        [bool((df[c].isna() == df["fecha_pedido"].isna()).all()) for c in TEMPORALES],
}, index=TEMPORALES)

,tipo,mínimo,máximo,nulos,los nulos coinciden con las fechas nulas
mes_pedido,Int64,1,12,22000,True
trimestre_pedido,Int64,1,4,22000,True
dia_semana_pedido,Int64,0,6,22000,True
es_fin_semana_pedido,Int64,0,1,22000,True


In [20]:
# Variables de texto. El marcador 'sin comentario' se trata como vacío, no como una frase: si
# se dejara, longitud_comentario contaría 15 caracteres de un comentario que no existe.
PATRON_DEMORA = r"demor|tard[oó]|retras|se demor"
comentario = df["comentario_cliente"].fillna("").astype(str).str.strip()
comentario = comentario.mask(comentario.str.lower() == "sin comentario", "")

df["longitud_comentario"]     = comentario.str.len().astype(int)
df["num_palabras_comentario"] = comentario.str.split().str.len().fillna(0).astype(int)
df["tiene_comentario"]        = (comentario != "").astype(int)
df["menciona_demora"]         = comentario.str.lower().str.contains(PATRON_DEMORA, regex=True).astype(int)
TEXTO = ["longitud_comentario", "num_palabras_comentario", "tiene_comentario", "menciona_demora"]
del comentario

pd.DataFrame({"valor": [
    bool((df["tiene_comentario"] == (~df["sin_comentario"]).astype(int)).all()),
    int(df.loc[df["tiene_comentario"] == 0, ["longitud_comentario",
                                             "num_palabras_comentario"]].sum(axis=1).ne(0).sum()),
    int(df["menciona_demora"].sum()), round(100 * float(df["menciona_demora"].mean()), 2),
]}, index=["tiene_comentario es el inverso de la bandera sin_comentario",
           "vacíos con longitud distinta de cero (debe ser 0)",
           "pedidos que mencionan demora", "% de pedidos que mencionan demora"])

,valor
tiene_comentario es el inverso de la bandera sin_comentario,True
vacíos con longitud distinta de cero (debe ser 0),0
pedidos que mencionan demora,44140
% de pedidos que mencionan demora,7.12


In [21]:
# Discretización de la edad en cinco rangos con sentido de negocio, más una categoría explícita
# para los clientes sin edad (que la Etapa 2 anuló por centinela).
BINS_EDAD = [17, 25, 35, 50, 65, 120]
ETIQUETAS_EDAD = ["18 a 25 (joven)", "26 a 35 (adulto joven)", "36 a 50 (adulto)",
                  "51 a 65 (adulto maduro)", "66 o más (adulto mayor)"]

df["grupo_edad"] = pd.cut(df["edad_cliente"], bins=BINS_EDAD, labels=ETIQUETAS_EDAD, right=True)
df["grupo_edad"] = df["grupo_edad"].cat.add_categories(["Sin dato"]).fillna("Sin dato")

resumen_edad = df.groupby("grupo_edad", observed=False)["edad_cliente"].agg(["size", "min", "max"])
resumen_edad.columns = ["clientes", "edad mínima", "edad máxima"]
resumen_edad

,clientes,edad mínima,edad máxima
grupo_edad,,,
18 a 25 (joven),80354,18.0,25.0
26 a 35 (adulto joven),100772,26.0,35.0
36 a 50 (adulto),151046,36.0,50.0
51 a 65 (adulto maduro),150946,51.0,65.0
66 o más (adulto mayor),100882,66.0,75.0
Sin dato,36000,NaN,NaN


In [22]:
# Corrección del sesgo del monto. log1p admite el cero sin trampa; Box-Cox exige valores
# estrictamente positivos, así que se aplica sobre monto + 1 y su lambda se aprende solo con
# entrenamiento.
es_train = df["conjunto"] == "entrenamiento"
desplazado = (df["monto_compra"] + 1.0).to_frame("monto_mas_uno")
valido = desplazado["monto_mas_uno"].notna()

df["monto_compra_log"] = np.log1p(df["monto_compra"])

boxcox = PowerTransformer(method="box-cox", standardize=False)
boxcox.fit(desplazado.loc[es_train & valido])                              # ajuste: solo entrenamiento
df["monto_compra_boxcox"] = np.nan
df.loc[valido, "monto_compra_boxcox"] = boxcox.transform(desplazado.loc[valido]).ravel()
del desplazado
gc.collect()

VERSIONES = ["monto_compra", "monto_compra_log", "monto_compra_boxcox"]
pd.DataFrame({
    "sesgo (todos los pedidos)": [round(float(df[c].skew()), 4) for c in VERSIONES],
    "sesgo (sin montos en cero)": [round(float(df.loc[df["monto_compra"] > 0, c].skew()), 4)
                                   for c in VERSIONES],
    "nulos": [int(df[c].isna().sum()) for c in VERSIONES],
}, index=VERSIONES).assign(**{"lambda de Box-Cox": [None, None, round(float(boxcox.lambdas_[0]), 6)]})

,sesgo (todos los pedidos),sesgo (sin montos en cero),nulos,lambda de Box-Cox
monto_compra,1.9978,2.0081,79291,NaN
monto_compra_log,-5.3167,-0.1721,79291,NaN
monto_compra_boxcox,0.0678,0.5629,79291,0.328597


---
## Diccionario de datos

Una fila por cada columna nueva creada entre las semanas 6 y 10. Las 69 columnas one-hot se
describen como familia, porque todas siguen la misma regla de construcción y nombrarlas una por una
haría la tabla ilegible sin agregar información.

In [23]:
DICCIONARIO = [
    ("fecha_pedido_invalida", "Semana 6", "Bandera: la fecha original tenía formato válido pero era una fecha de calendario imposible (por ejemplo 31/02). La fila se conserva; la fecha queda vacía."),
    ("precio_unitario_atipico", "Semana 7", "Bandera: el precio original era negativo o uno de los dos centinelas (999.000.000 o 50.000.000). El valor se anuló."),
    ("unidades_vendidas_atipico", "Semana 7", "Bandera: las unidades eran negativas o uno de los dos centinelas (9999 o 5000). El valor se anuló."),
    ("edad_cliente_atipico", "Semana 7", "Bandera: la edad era negativa, cero, o uno de los dos centinelas (200 o 150). El valor se anuló."),
    ("monto_atipico", "Semana 7", "Bandera: el monto se quedó sin valor al recalcularlo, porque uno de sus dos factores fue anulado."),
    ("peso_pedido_kg_original", "Semana 7", "Copia del peso antes de capar, para poder rastrear cuánto se movió cada valor."),
    ("peso_capado", "Semana 7", "Bandera: el peso superaba el percentil 99 y quedó recortado a ese tope."),
    ("conjunto", "Semana 8", "Partición aleatoria con semilla 42: 'entrenamiento' (80 %) o 'prueba' (20 %). Define qué filas ajustan los escaladores y codificadores."),
    ("precio_unitario_esc", "Semana 8", "precio_unitario escalado con Min-Max ajustado solo en entrenamiento; queda entre 0 y 1."),
    ("unidades_vendidas_esc", "Semana 8", "unidades_vendidas escalada con Min-Max ajustado solo en entrenamiento."),
    ("monto_compra_esc", "Semana 8", "monto_compra escalado con escalado robusto (mediana e IQR de entrenamiento), porque tiene atípicos legítimos."),
    ("edad_cliente_esc", "Semana 8", "edad_cliente escalada con Z-score (media y desviación de entrenamiento); la distribución es simétrica."),
    ("peso_pedido_kg_esc", "Semana 8", "peso_pedido_kg escalado con escalado robusto, porque es bimodal y tiene atípicos."),
    ("dominio_correo", "Semana 9", "Dominio extraído de correo_cliente: es la única categoría real escondida dentro de un identificador. 'desconocido' cuando no hay correo."),
    ("canal_compra_*, metodo_pago_*, categoria_producto_*, ciudad_tienda_*, comentario_cliente_*, dominio_correo_*, producto_*", "Semana 9", "69 columnas one-hot, una por categoría, en 0 y 1. Las categorías las aprendió el codificador solo con entrenamiento."),
    ("nivel_lealtad_ord", "Semana 9", "Nivel de lealtad codificado con su orden real: Bronce=0, Plata=1, Oro=2, Platino=3. Alfabéticamente el orden se rompería."),
    ("nivel_satisfaccion_ord", "Semana 9", "Nivel de satisfacción codificado con su orden real: Bajo=0, Medio=1, Alto=2. Nulo cuando no hay dato."),
    ("codigo_postal_freq", "Semana 9", "Proporción de pedidos de cada código postal en el conjunto de entrenamiento. Dice si la zona es de mucho o poco volumen."),
    ("codigo_postal_te", "Semana 9", "Target encoding del código postal agrupado contra fuga_cliente, con suavizado y cross fitting de 5 pliegues para que ninguna fila use su propio objetivo."),
    ("mes_pedido", "Semana 10", "Mes del pedido, de 1 a 12, extraído de fecha_pedido. Nulo si el pedido no tiene fecha."),
    ("trimestre_pedido", "Semana 10", "Trimestre del pedido, de 1 a 4."),
    ("dia_semana_pedido", "Semana 10", "Día de la semana, de 0 (lunes) a 6 (domingo)."),
    ("es_fin_semana_pedido", "Semana 10", "1 si el pedido cayó sábado o domingo, 0 si no."),
    ("longitud_comentario", "Semana 10", "Número de caracteres del comentario. Cero cuando no hay comentario."),
    ("num_palabras_comentario", "Semana 10", "Número de palabras del comentario. Cero cuando no hay comentario."),
    ("tiene_comentario", "Semana 10", "1 si el pedido tiene comentario real, 0 si el campo trae el marcador 'sin comentario'."),
    ("menciona_demora", "Semana 10", "1 si el comentario menciona una demora (patrón demor|tardó|retras). Es la queja que más se repite."),
    ("grupo_edad", "Semana 10", "edad_cliente discretizada en cinco rangos con sentido de negocio (18-25, 26-35, 36-50, 51-65, 66+), más 'Sin dato' para los clientes cuya edad se anuló en la Etapa 2."),
    ("monto_compra_log", "Semana 10", "Logaritmo natural de monto_compra + 1, para reducir su sesgo. Admite montos en cero sin trampa."),
    ("monto_compra_boxcox", "Semana 10", "monto_compra + 1 transformado con Box-Cox, con el lambda aprendido solo en entrenamiento. Es la corrección de sesgo más fuerte de las dos."),
]

diccionario_datos = pd.DataFrame(DICCIONARIO,
                                 columns=["columna del dataset", "de qué semana viene", "qué significa"])
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 40):
    display(diccionario_datos.set_index("columna del dataset"))

,de qué semana viene,qué significa
columna del dataset,,
fecha_pedido_invalida,Semana 6,Bandera: la fecha original tenía formato válido pero era una fecha de calendario imposible (por ejemplo 31/02). La fila se conserva; la fecha queda vacía.
precio_unitario_atipico,Semana 7,Bandera: el precio original era negativo o uno de los dos centinelas (999.000.000 o 50.000.000). El valor se anuló.
unidades_vendidas_atipico,Semana 7,Bandera: las unidades eran negativas o uno de los dos centinelas (9999 o 5000). El valor se anuló.
edad_cliente_atipico,Semana 7,"Bandera: la edad era negativa, cero, o uno de los dos centinelas (200 o 150). El valor se anuló."
monto_atipico,Semana 7,"Bandera: el monto se quedó sin valor al recalcularlo, porque uno de sus dos factores fue anulado."
peso_pedido_kg_original,Semana 7,"Copia del peso antes de capar, para poder rastrear cuánto se movió cada valor."
peso_capado,Semana 7,Bandera: el peso superaba el percentil 99 y quedó recortado a ese tope.
conjunto,Semana 8,Partición aleatoria con semilla 42: 'entrenamiento' (80 %) o 'prueba' (20 %). Define qué filas ajustan los escaladores y codificadores.
precio_unitario_esc,Semana 8,precio_unitario escalado con Min-Max ajustado solo en entrenamiento; queda entre 0 y 1.


---
## Exportación y control de calidad final

Un solo archivo de salida. Las tres comprobaciones que el comité pidió: que no queden duplicados,
que estén las columnas de las cinco etapas, y que el resultado coincida con el de la Semana 10.

In [ ]:
# El objetivo y la partición se dejan al final, como en todos los cuadernos de la misión.
df = df[[c for c in df.columns if c not in (OBJETIVO, "conjunto")] + [OBJETIVO, "conjunto"]]
df.to_csv(SALIDA, index=False)

COLUMNAS_POR_ETAPA = {
    "Etapa 1 · limpieza":        ["fecha_pedido_invalida"],
    "Etapa 2 · atípicos":        ["precio_unitario_atipico", "unidades_vendidas_atipico",
                                  "edad_cliente_atipico", "monto_atipico", "peso_capado"],
    "Etapa 3 · escalado":        [c + "_esc" for c in PLAN],
    "Etapa 4 · codificación":    ["nivel_lealtad_ord", "nivel_satisfaccion_ord",
                                  "codigo_postal_freq", "codigo_postal_te"],
    "Etapa 5 · transformaciones": TEMPORALES + TEXTO + ["grupo_edad", "monto_compra_log",
                                                        "monto_compra_boxcox"],
}

control = pd.DataFrame({
    "columnas esperadas": [len(v) for v in COLUMNAS_POR_ETAPA.values()],
    "columnas presentes": [sum(c in df.columns for c in v) for v in COLUMNAS_POR_ETAPA.values()],
}, index=list(COLUMNAS_POR_ETAPA))
control["todas presentes"] = control["columnas esperadas"] == control["columnas presentes"]
display(control)

pd.DataFrame({"valor": [
    SALIDA, f"{os.path.getsize(SALIDA) / 1048576:.1f} MB",
    filas_inicio, len(df), df.shape[1],
    int(df.duplicated().sum()),
    int((df["conjunto"] == "entrenamiento").sum()), int((df["conjunto"] == "prueba").sum()),
    bool(control["todas presentes"].all()),
]}, index=["archivo de salida", "tamaño", "filas de entrada", "filas de salida",
           "columnas de salida", "duplicados exactos (debe ser 0)",
           "filas de entrenamiento", "filas de prueba",
           "las cinco etapas dejaron sus columnas"])

In [ ]:
# Comprobación contra la entrega de la semana pasada. Si el archivo de la Semana 10 no está en
# la carpeta, la celda lo dice y sigue: el pipeline no depende de él, solo se compara con él.
REFERENCIA = "S10_PD_Borja_DatasetTransformado.csv"
if os.path.exists(REFERENCIA):
    cabecera = pd.read_csv(REFERENCIA, nrows=0)
    filas_ref = sum(1 for _ in open(REFERENCIA, encoding="utf-8")) - 1
    faltan = [c for c in cabecera.columns if c not in df.columns]
    sobran = [c for c in df.columns if c not in cabecera.columns]
    comparacion = pd.DataFrame({"valor": [
        filas_ref, len(df), filas_ref == len(df),
        cabecera.shape[1], df.shape[1], cabecera.shape[1] == df.shape[1],
        ", ".join(faltan) or "ninguna", ", ".join(sobran) or "ninguna",
    ]}, index=["filas en la Semana 10", "filas en este pipeline", "coinciden las filas",
               "columnas en la Semana 10", "columnas en este pipeline", "coinciden las columnas",
               "columnas de la Semana 10 que faltan aquí", "columnas nuevas que no estaban allá"])
else:
    comparacion = pd.DataFrame({"valor": [f"{REFERENCIA} no está en la carpeta; se omite la comparación"]},
                               index=["aviso"])

with pd.option_context("display.max_colwidth", 160):
    display(comparacion)

---
### Cierre

El pipeline entra por un archivo y sale por otro, sin pasos manuales entre etapas. El dataset
`S11_PD_Grupo05_DatasetTransformado.csv` es el insumo de la **Misión 3, Ingeniería de características
e integración**: sobre él se construyen las variables derivadas del cliente y se integran las
fuentes externas.